In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))


GPU Available: True
GPU Device Name: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
import torch

# Create a tensor and move it to the GPU
device = torch.device("cuda")
x = torch.ones(3, 3, device=device)

# Perform a quick operation on the GPU
y = x * 2
print("Tensor location:", y.device)
print("Result:\n", y)
from pathlib import Path
import sys
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.autoencoder_config import AutoEncoderConfig
from src.models.autoencoder import AutoEncoderModel

Tensor location: cuda:0
Result:
 tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')


In [3]:
from pathlib import Path
import sys
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.autoencoder_config import AutoEncoderConfig
from src.models.autoencoder import AutoEncoderModel

In [4]:
from src.preprocessing.loader import DataLoader

loader = DataLoader()

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", 
]

df = loader.load_multiple(files)

print(df.shape)

2026-08-09 11:13:47 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv started.
2026-08-09 11:13:47 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Monday-WorkingHours.pcap_ISCX.csv
2026-08-09 11:13:50 | INFO     | AdaptiveRL | Loaded Monday-WorkingHours.pcap_ISCX.csv | Shape=(529918, 79)
2026-08-09 11:13:50 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv completed in 2.7337 seconds.
2026-08-09 11:13:50 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv started.
2026-08-09 11:13:50 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Tuesday-WorkingHours.pcap_ISCX.csv
2026-08-09 11:13:52 | INFO     | AdaptiveRL | Loaded Tuesday-WorkingHours.pcap_ISCX.csv | Shape=(445909, 79)
2026-08-09 11:13:52 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv completed in 1.9356 seconds.
2026-08-09 11:13:52 | INFO     | AdaptiveRL | Lo

(2830743, 79)


In [5]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()

df = cleaner.clean(df)

print(df.shape)

2026-08-09 11:14:10 | INFO     | AdaptiveRL | Replacing Infinite Values started.
2026-08-09 11:14:14 | INFO     | AdaptiveRL | Replacing Infinite Values completed in 3.1958 seconds.
2026-08-09 11:14:14 | INFO     | AdaptiveRL | Removing Duplicates started.
2026-08-09 11:14:25 | INFO     | AdaptiveRL | Removed 308381 duplicate rows.
2026-08-09 11:14:25 | INFO     | AdaptiveRL | Removing Duplicates completed in 10.7937 seconds.
2026-08-09 11:14:25 | INFO     | AdaptiveRL | Removing Missing Values started.
2026-08-09 11:14:26 | INFO     | AdaptiveRL | Removed 1564 rows containing missing values.
2026-08-09 11:14:26 | INFO     | AdaptiveRL | Removing Missing Values completed in 0.8698 seconds.
2026-08-09 11:14:26 | INFO     | AdaptiveRL | Removing Constant Columns started.
2026-08-09 11:14:28 | INFO     | AdaptiveRL | Removed 8 constant columns.
2026-08-09 11:14:28 | INFO     | AdaptiveRL | Removing Constant Columns completed in 2.0090 seconds.
2026-08-09 11:14:28 | INFO     | AdaptiveRL |

(2520798, 71)


In [6]:
df.columns = df.columns.str.strip()

In [7]:
from src.preprocessing.encoder import DataEncoder

encoder = DataEncoder(target_column=" Label")

df = encoder.fit_transform(df)

print(df.dtypes["Label"])
print(df["Label"].unique()[:10])

2026-08-09 11:14:39 | INFO     | AdaptiveRL | Encoding Dataset started.
2026-08-09 11:14:40 | INFO     | AdaptiveRL | Encoding Dataset completed in 0.3224 seconds.
2026-08-09 11:14:40 | INFO     | AdaptiveRL | Encoding completed.


int64
[ 0  7 11  6  5  4  3  8 12 14]


In [8]:
from src.preprocessing.scaler import DataScaler

scaler = DataScaler(
    method="standard",
    target_column="Label",
)

df = scaler.fit_transform(df)

print(df.shape)
print(df["Label"].dtype)
print(df["Label"].unique()[:10])

2026-08-09 11:14:53 | INFO     | AdaptiveRL | Scaler (standard) fitted on 70 feature columns.
2026-08-09 11:14:54 | INFO     | AdaptiveRL | Scaling Dataset started.
2026-08-09 11:14:56 | INFO     | AdaptiveRL | Scaling Dataset completed in 1.9386 seconds.
2026-08-09 11:14:56 | INFO     | AdaptiveRL | Scaling completed.


(2520798, 71)
int64
[ 0  7 11  6  5  4  3  8 12 14]


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Label"])
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2016638, 70)
X_test : (504160, 70)
y_train: (2016638,)
y_test : (504160,)


In [10]:
X_normal_train = X_train[y_train == 0]

print("Normal-only training shape:", X_normal_train.shape)
print("Normal samples:", len(X_normal_train))

Normal-only training shape: (1676045, 70)
Normal samples: 1676045


In [11]:
print("Unique labels used for AutoEncoder training:", y_train[y_train == 0].unique())

Unique labels used for AutoEncoder training: [0]


In [12]:
ae_config = AutoEncoderConfig(
    input_dim=X_train.shape[1]
)

print(ae_config)

AutoEncoderConfig(input_dim=70, hidden_dims=[64, 32], latent_dim=16, activation='relu', dropout=0.0, batch_size=512, epochs=50, learning_rate=0.001, weight_decay=0.0, reconstruction_loss='mse', validation_split=0.1, early_stopping_patience=5, early_stopping_min_delta=0.0001, threshold_strategy='percentile', threshold_percentile=95.0, threshold_std_multiplier=3.0, device='auto', random_seed=42, num_workers=0, checkpoint_dir=None, checkpoint_every_n_epochs=5, gradient_clip_norm=None, log_every_n_epochs=1)


In [13]:
autoencoder = AutoEncoderModel(ae_config)

print(autoencoder)
print("Device:", autoencoder.device)

2026-08-09 11:16:46 | INFO     | src.models.autoencoder | AutoEncoder initialized | device=cuda | input_dim=70 | hidden_dims=[64, 32] | latent_dim=16


AutoEncoderModel(input_dim=70, latent_dim=16, loss='mse', device='cuda', threshold=None, is_fitted=False)
Device: cuda


In [14]:
autoencoder.fit(X_normal_train)

2026-08-09 11:16:57 | INFO     | src.models.autoencoder | Training AutoEncoder | train_samples=1508441 | val_samples=167604 | features=70 | batch_size=512 | epochs=50
2026-08-09 11:17:16 | INFO     | src.models.autoencoder | Epoch 1/50 | train_loss=0.22552057 | validation_loss=0.12416526
2026-08-09 11:17:29 | INFO     | src.models.autoencoder | Epoch 2/50 | train_loss=0.15232287 | validation_loss=0.05276386
2026-08-09 11:17:42 | INFO     | src.models.autoencoder | Epoch 3/50 | train_loss=0.11078316 | validation_loss=0.15887208
2026-08-09 11:17:55 | INFO     | src.models.autoencoder | Epoch 4/50 | train_loss=0.10811218 | validation_loss=0.06315791
2026-08-09 11:18:09 | INFO     | src.models.autoencoder | Epoch 5/50 | train_loss=0.09686335 | validation_loss=0.06601464
2026-08-09 11:18:22 | INFO     | src.models.autoencoder | Epoch 6/50 | train_loss=0.09934055 | validation_loss=0.07253403
2026-08-09 11:18:34 | INFO     | src.models.autoencoder | Epoch 7/50 | train_loss=0.08718929 | valida

AutoEncoderModel(input_dim=70, latent_dim=16, loss='mse', device='cuda', threshold=0.07559069246053696, is_fitted=True)

In [16]:
from pathlib import Path

save_dir = Path("trained_models")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "autoencoder.joblib"

autoencoder.save(model_path)

print(f"Model saved to: {model_path}")

2026-08-09 11:22:30 | INFO     | src.models.autoencoder | AutoEncoder saved -> trained_models/autoencoder.joblib


Model saved to: trained_models/autoencoder.joblib


In [17]:
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTest label distribution:")
print(y_test.value_counts().sort_index())

X_test: (504160, 70)
y_test: (504160,)

Test label distribution:
Label
0     419012
1        390
2      25603
3       2057
4      34569
5       1046
6       1077
7       1186
8          2
9          7
10     18139
11       644
12       294
13         4
14       130
Name: count, dtype: int64


In [18]:
autoencoder = AutoEncoderModel.load(MODEL_PATH)

print(autoencoder)

2026-08-09 11:23:11 | INFO     | src.models.autoencoder | AutoEncoder loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/trained_models/autoencoder.joblib


AutoEncoderModel(input_dim=70, latent_dim=16, loss='mse', device='cuda', threshold=0.07559069246053696, is_fitted=True)


In [19]:
ae_predictions = autoencoder.predict(X_test)

ae_scores = autoencoder.anomaly_score(X_test)

print("Predictions shape:", ae_predictions.shape)
print("Scores shape:", ae_scores.shape)

print("\nPredicted label distribution:")
print(np.unique(ae_predictions, return_counts=True))

Predictions shape: (504160,)
Scores shape: (504160,)

Predicted label distribution:
(array([0, 1], dtype=int8), array([434311,  69849]))


In [21]:
from src.evaluation.metrics import MetricsEvaluator

# Convert CICIDS2017 labels:
# 0 = BENIGN
# anything else = ATTACK
y_test_binary = (y_test != 0).astype(int)

print("Binary test-label distribution:")
print(np.unique(y_test_binary, return_counts=True))

evaluator = MetricsEvaluator()

ae_result = evaluator.evaluate(
    y_true=y_test_binary,
    y_pred=ae_predictions,
    anomaly_scores=ae_scores,
    model_name="AutoEncoder",
)

print(ae_result.summary())

Binary test-label distribution:
(array([0, 1]), array([419012,  85148]))
Evaluation Summary: AutoEncoder
Samples          : 504160 (Normal: 419012, Anomaly: 85148)
------------------------------------------------------------
Accuracy         : 0.8862
Precision        : 0.6988
Recall           : 0.5733
F1 Score         : 0.6298
ROC-AUC          : 0.8984
PR-AUC           : 0.7311
------------------------------------------------------------
True Positive Rate  (TPR) : 0.5733
True Negative Rate  (TNR) : 0.9498
False Positive Rate (FPR) : 0.0502
False Negative Rate (FNR) : 0.4267
------------------------------------------------------------
Confusion Matrix:
    TN: 397975   FP: 21037   
    FN: 36336    TP: 48812   


In [25]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())

evaluation_results = joblib.load(
    RESULTS_DIR / "evaluation_results.joblib"
)

print("\nSaved result keys:")
print(evaluation_results.keys())

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection
Results directory: /home/kalpe/projects/adaptive_rl_anomaly_detection/evaluation_results
Exists: True

Saved result keys:
dict_keys(['if_result', 'lof_result', 'svm_result'])


In [26]:
if_result = evaluation_results["if_result"]
lof_result = evaluation_results["lof_result"]
svm_result = evaluation_results["svm_result"]

print("Baseline results loaded:")
print("✓ Isolation Forest")
print("✓ Local Outlier Factor")
print("✓ One-Class SVM")

Baseline results loaded:
✓ Isolation Forest
✓ Local Outlier Factor
✓ One-Class SVM


In [27]:
from src.evaluation.metrics import MetricsEvaluator

evaluator = MetricsEvaluator()

comparison_df = evaluator.compare(
    [
        if_result,
        lof_result,
        svm_result,
        ae_result,
    ]
)

print(comparison_df.to_string(index=False))

          model_name  accuracy  precision   recall       f1  roc_auc   pr_auc  true_positive_rate  true_negative_rate  false_positive_rate  false_negative_rate  true_negatives  false_positives  false_negatives  true_positives  n_samples  n_anomalies  n_normal notes
    Isolation Forest  0.815386   0.343828 0.102480 0.157898 0.749923 0.339947            0.102480            0.960257             0.039743             0.897520          402359            16653            76422            8726     504160        85148    419012      
Local Outlier Factor  0.790541   0.100979 0.030394 0.046725 0.458718 0.150706            0.030394            0.945011             0.054989             0.969606          395971            23041            82560            2588     504160        85148    419012      
       One-Class SVM  0.807077   0.260799 0.077571 0.119576 0.722146 0.304867            0.077571            0.955321             0.044679             0.922429          400291            18721          

In [28]:
comparison_df[
    [
        "model_name",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
    ]
]

,model_name,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Isolation Forest,0.815386,0.343828,0.102480,0.157898,0.749923,0.339947
1,Local Outlier Factor,0.790541,0.100979,0.030394,0.046725,0.458718,0.150706
2,One-Class SVM,0.807077,0.260799,0.077571,0.119576,0.722146,0.304867
3,AutoEncoder,0.886201,0.698822,0.573261,0.629844,0.898446,0.731135


In [29]:
from pathlib import Path
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"

evaluation_results = joblib.load(
    RESULTS_DIR / "evaluation_results.joblib"
)

model_predictions = joblib.load(
    RESULTS_DIR / "model_predictions.joblib"
)

model_scores = joblib.load(
    RESULTS_DIR / "model_scores.joblib"
)

metrics_df = pd.read_csv(
    RESULTS_DIR / "metrics.csv"
)

benchmark_df = pd.read_csv(
    RESULTS_DIR / "benchmark.csv"
)

print("Evaluation results:", evaluation_results.keys())
print("Predictions:", model_predictions.keys())
print("Scores:", model_scores.keys())

print("\nMetrics shape:", metrics_df.shape)
print("Benchmark shape:", benchmark_df.shape)

Evaluation results: dict_keys(['if_result', 'lof_result', 'svm_result'])
Predictions: dict_keys(['if_predictions', 'lof_predictions', 'svm_predictions'])
Scores: dict_keys(['if_scores', 'lof_scores', 'svm_scores'])

Metrics shape: (3, 19)
Benchmark shape: (3, 3)


In [30]:
evaluation_results["ae_result"] = ae_result

joblib.dump(
    evaluation_results,
    RESULTS_DIR / "evaluation_results.joblib",
)

print("✓ Added AutoEncoder evaluation result")

✓ Added AutoEncoder evaluation result


In [31]:
model_predictions["ae_predictions"] = ae_predictions

joblib.dump(
    model_predictions,
    RESULTS_DIR / "model_predictions.joblib",
)

print("✓ Added AutoEncoder predictions")

✓ Added AutoEncoder predictions


In [32]:
model_scores["ae_scores"] = ae_scores

joblib.dump(
    model_scores,
    RESULTS_DIR / "model_scores.joblib",
)

print("✓ Added AutoEncoder scores")

✓ Added AutoEncoder scores


In [33]:
comparison_df.to_csv(
    RESULTS_DIR / "metrics.csv",
    index=False,
)

print("✓ Updated metrics.csv")

✓ Updated metrics.csv


In [34]:
pd.read_csv(
    RESULTS_DIR / "metrics.csv"
)

,model_name,accuracy,precision,recall,f1,roc_auc,pr_auc,true_positive_rate,true_negative_rate,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,n_samples,n_anomalies,n_normal,notes
0,Isolation Forest,0.815386,0.343828,0.102480,0.157898,0.749923,0.339947,0.102480,0.960257,0.039743,0.897520,402359,16653,76422,8726,504160,85148,419012,NaN
1,Local Outlier Factor,0.790541,0.100979,0.030394,0.046725,0.458718,0.150706,0.030394,0.945011,0.054989,0.969606,395971,23041,82560,2588,504160,85148,419012,NaN
2,One-Class SVM,0.807077,0.260799,0.077571,0.119576,0.722146,0.304867,0.077571,0.955321,0.044679,0.922429,400291,18721,78543,6605,504160,85148,419012,NaN
3,AutoEncoder,0.886201,0.698822,0.573261,0.629844,0.898446,0.731135,0.573261,0.949794,0.050206,0.426739,397975,21037,36336,48812,504160,85148,419012,NaN


In [35]:
import time

# Prediction benchmark
start = time.perf_counter()

_ = autoencoder.predict(X_test)

ae_prediction_time = time.perf_counter() - start


# Scoring benchmark
start = time.perf_counter()

_ = autoencoder.anomaly_score(X_test)

ae_scoring_time = time.perf_counter() - start


print(f"Prediction time: {ae_prediction_time:.6f} s")
print(f"Scoring time:    {ae_scoring_time:.6f} s")

Prediction time: 4.962989 s
Scoring time:    4.233321 s


In [36]:
n_test = len(X_test)

ae_prediction_throughput = (
    n_test / ae_prediction_time
)

ae_scoring_throughput = (
    n_test / ae_scoring_time
)

print(
    f"Prediction throughput: "
    f"{ae_prediction_throughput:.2f} samples/sec"
)

print(
    f"Scoring throughput: "
    f"{ae_scoring_throughput:.2f} samples/sec"
)

Prediction throughput: 101583.94 samples/sec
Scoring throughput: 119093.27 samples/sec


In [37]:
print(benchmark_df.columns.tolist())
print(benchmark_df)

['model_name', 'prediction_time_seconds', 'scoring_time_seconds']
             model_name  prediction_time_seconds  scoring_time_seconds
0      Isolation Forest                 3.450430              3.550077
1  Local Outlier Factor                33.253419             37.121474
2         One-Class SVM                71.715006             71.512330
